# FD-24: Полный цикл экспериментов M2 (Слоты 1–9) — Kaggle 2×T4
> **Режим запуска:** Рекомендуется запуск единым блокнотом в одной интерактивной сессии Kaggle или Save & Run All (~4.5–5.0 ч).
> **Инвариант выбора лучшего:** Строго по max PR-AUC на реальных строках `val` (не по тестам и не по синтетике).
> **Ограничение диска:** Промежуточные чекпоинты (`checkpoint-*`) удаляются сразу после обучения слота, сохраняются только `best/` и артефакты оценки (`preds.csv`, `metrics.json`, `table_e1.csv`).

In [ ]:
%cd /kaggle/working
import os
from kaggle_secrets import UserSecretsClient
try:
    _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    _repo = f"https://{_tok}@github.com/o1uh/FreshDoc.git"
    print("clone: private (token)")
except Exception:
    _repo = "https://github.com/o1uh/FreshDoc.git"
    print("clone: public fallback (repo must be public)")

!if [ ! -d FreshDoc ]; then git clone --quiet {_repo} > /tmp/clone.log 2>&1 || (echo "CLONE FAILED:"; sed -E 's#(https?://)[^@]+@#\1***@#g' /tmp/clone.log); fi
if '_tok' in locals(): del _tok
if '_repo' in locals(): del _repo

%cd /kaggle/working/FreshDoc
!git checkout feat/FD-24-kaggle-experiments 2>/dev/null || !git checkout main 2>/dev/null || true
!git pull --ff-only || echo "PULL SKIPPED: убедись, что код свежий"

import os
os.environ["NCCL_P2P_DISABLE"] = "1"
!python -c "import transformers, sys; from packaging import version; sys.exit(0 if version.parse(transformers.__version__) >= version.parse('4.48.0') else 1)" || pip install --quiet --no-deps "transformers>=4.48.0"
!nvidia-smi

In [ ]:
import os, glob
import pandas as pd
_cands = glob.glob("/kaggle/input/**/dataset_v1.parquet", recursive=True)
assert _cands, "dataset_v1.parquet не найден в /kaggle/input (добавь датасет в Inputs блокнота)"
_p = _cands[0]
os.environ["DATA_PATH"] = _p
print(f"Dataset path: {_p}")

_df = pd.read_parquet(_p, columns=["split"])
assert len(_df) == 27203, f"dataset rows {len(_df)} != 27203"
_exp = {"train": 21997, "val": 559, "test_real": 2895, "test_synth": 1552, "gold": 200}
_got = _df["split"].value_counts().to_dict()
assert _got == _exp, f"split mismatch: {_got}"
print("dataset OK:", _got)
del _df

In [ ]:
# Генерация бейзлайнов M0 и M1 (подготовка для evaluate и стэкинга)
!python scripts/baselines_m0_m1.py --dataset "$DATA_PATH" \
  --out_m0 /kaggle/working/m0.csv --out_m1 /kaggle/working/m1.csv \
  --out_model /kaggle/working/m1_hgb.joblib --out_report /kaggle/working/m1_report.json

## Шаг 1: Предполётная проверка бэкбонов (workflow.md §6 FD-24, шаг 1)

In [ ]:
# Предполётка: smoke-тест весов (granite 48M, UniXcoder, ModernBERT, GTE)
import gc, torch
from transformers import AutoModel, AutoTokenizer
for m in ["ibm-granite/granite-embedding-small-english-r2",
          "microsoft/unixcoder-base",
          "answerdotai/ModernBERT-base",
          "Alibaba-NLP/gte-modernbert-base"]:
    tok = AutoTokenizer.from_pretrained(m)
    mod = AutoModel.from_pretrained(m).to("cuda")
    out = mod(**tok("def test(): pass", return_tensors="pt").to("cuda"))
    print(f"Smoke OK: {m:45s} shape={tuple(out.last_hidden_state.shape)}")
    del mod, out, tok
    gc.collect()
    torch.cuda.empty_cache()
print("GPU freed.")


In [ ]:
# Предполётка: тест на переобучение ModernBERT (200 строк x 20 эпох)
# Ожидание: PR-AUC >= 0.99 (критерий R10)
!python scripts/train_m2.py --config config/train.yaml \
  --data_path "$DATA_PATH" --model_name answerdotai/ModernBERT-base \
  --output_dir /kaggle/working/checkpoints/pilot_modernbert_overfit --overfit_test
!rm -rf /kaggle/working/checkpoints/pilot_modernbert_overfit

In [ ]:
# Предполётка @1024: пилот 100 шагов FP16 на ModernBERT с проверкой NaN и замером с/шаг
# Если уходит в NaN — стоп-критерий: снизить lr до 1e-5 (--lr 1e-5)
!torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml \
  --data_path "$DATA_PATH" --model_name answerdotai/ModernBERT-base \
  --max_len 1024 --pilot_steps 100 --output_dir /kaggle/working/checkpoints/pilot_modernbert_100
!rm -rf /kaggle/working/checkpoints/pilot_modernbert_100

# Предполётка @2048: ранняя валидация аллокации памяти и исключение риска CUDA OOM
!torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml \
  --data_path "$DATA_PATH" --model_name Alibaba-NLP/gte-modernbert-base \
  --max_len 2048 --gradient_checkpointing --pilot_steps 20 --output_dir /kaggle/working/checkpoints/pilot_gte_2048
!rm -rf /kaggle/working/checkpoints/pilot_gte_2048


## Сессия 1: Эксперимент E2 — Сравнение смесей данных (Слоты 1–4)

In [ ]:
# ==============================================================================
# Слот 1: E2(a) Только реальные данные (2 393 train)
# ==============================================================================
# 1. Обучение модели
!torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml \
  --data_path "$DATA_PATH" --train_mix real_only \
  --output_dir /kaggle/working/checkpoints/slot1_real --runs_csv reports/runs.csv

# 2. Очистка промежуточных чекпоинтов (сохранение дисковой квоты 20 ГБ)
!rm -rf /kaggle/working/checkpoints/slot1_real/checkpoint-*
!ls -lh /kaggle/working/checkpoints/slot1_real/best

# 3. Инференс на test_real и gold (проверка доменного сдвига E2 / вклада diff E3)
!python scripts/predict_m2.py --checkpoint /kaggle/working/checkpoints/slot1_real/best \
  --dataset "$DATA_PATH"  --out /kaggle/working/checkpoints/slot1_real/preds.csv

# 4. Полная оценка (metrics.json + таблица e1.csv по сплитам val, test_real, gold)
!python scripts/evaluate.py --dataset "$DATA_PATH" \
  --pred M0=/kaggle/working/m0.csv --pred M1=/kaggle/working/m1.csv --pred M2=/kaggle/working/checkpoints/slot1_real/preds.csv \
  --out_metrics /kaggle/working/checkpoints/slot1_real/metrics.json --out_table /kaggle/working/checkpoints/slot1_real/table_e1.csv \
  --figdir /kaggle/working/checkpoints/slot1_real/figures_e1
!head -n 25 /kaggle/working/checkpoints/slot1_real/table_e1.csv

In [ ]:
# ==============================================================================
# Слот 2: E2(b) Только синтетика AST (19 008 train, без LLM)
# ==============================================================================
# 1. Обучение модели
!torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml \
  --data_path "$DATA_PATH" --train_mix synth_only \
  --output_dir /kaggle/working/checkpoints/slot2_synth --runs_csv reports/runs.csv

# 2. Очистка промежуточных чекпоинтов (сохранение дисковой квоты 20 ГБ)
!rm -rf /kaggle/working/checkpoints/slot2_synth/checkpoint-*
!ls -lh /kaggle/working/checkpoints/slot2_synth/best

# 3. Инференс на test_real и gold (проверка доменного сдвига E2 / вклада diff E3)
!python scripts/predict_m2.py --checkpoint /kaggle/working/checkpoints/slot2_synth/best \
  --dataset "$DATA_PATH"  --out /kaggle/working/checkpoints/slot2_synth/preds.csv

# 4. Полная оценка (metrics.json + таблица e1.csv по сплитам val, test_real, gold)
!python scripts/evaluate.py --dataset "$DATA_PATH" \
  --pred M0=/kaggle/working/m0.csv --pred M1=/kaggle/working/m1.csv --pred M2=/kaggle/working/checkpoints/slot2_synth/preds.csv \
  --out_metrics /kaggle/working/checkpoints/slot2_synth/metrics.json --out_table /kaggle/working/checkpoints/slot2_synth/table_e1.csv \
  --figdir /kaggle/working/checkpoints/slot2_synth/figures_e1
!head -n 25 /kaggle/working/checkpoints/slot2_synth/table_e1.csv

In [ ]:
# ==============================================================================
# Слот 3: E2(c) Реальные данные + AST (21 401 train)
# ==============================================================================
# 1. Обучение модели
!torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml \
  --data_path "$DATA_PATH" --train_mix real_ast \
  --output_dir /kaggle/working/checkpoints/slot3_real_ast --runs_csv reports/runs.csv

# 2. Очистка промежуточных чекпоинтов (сохранение дисковой квоты 20 ГБ)
!rm -rf /kaggle/working/checkpoints/slot3_real_ast/checkpoint-*
!ls -lh /kaggle/working/checkpoints/slot3_real_ast/best

# 3. Инференс на test_real и gold (проверка доменного сдвига E2 / вклада diff E3)
!python scripts/predict_m2.py --checkpoint /kaggle/working/checkpoints/slot3_real_ast/best \
  --dataset "$DATA_PATH"  --out /kaggle/working/checkpoints/slot3_real_ast/preds.csv

# 4. Полная оценка (metrics.json + таблица e1.csv по сплитам val, test_real, gold)
!python scripts/evaluate.py --dataset "$DATA_PATH" \
  --pred M0=/kaggle/working/m0.csv --pred M1=/kaggle/working/m1.csv --pred M2=/kaggle/working/checkpoints/slot3_real_ast/preds.csv \
  --out_metrics /kaggle/working/checkpoints/slot3_real_ast/metrics.json --out_table /kaggle/working/checkpoints/slot3_real_ast/table_e1.csv \
  --figdir /kaggle/working/checkpoints/slot3_real_ast/figures_e1
!head -n 25 /kaggle/working/checkpoints/slot3_real_ast/table_e1.csv

In [ ]:
# ==============================================================================
# Слот 4: E2(d) Полная смесь (+ LLM-пары, 21 997 train)
# ==============================================================================
# 1. Обучение модели
!torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml \
  --data_path "$DATA_PATH" --train_mix full \
  --output_dir /kaggle/working/checkpoints/slot4_full --runs_csv reports/runs.csv

# 2. Очистка промежуточных чекпоинтов (сохранение дисковой квоты 20 ГБ)
!rm -rf /kaggle/working/checkpoints/slot4_full/checkpoint-*
!ls -lh /kaggle/working/checkpoints/slot4_full/best

# 3. Инференс на test_real и gold (проверка доменного сдвига E2 / вклада diff E3)
!python scripts/predict_m2.py --checkpoint /kaggle/working/checkpoints/slot4_full/best \
  --dataset "$DATA_PATH"  --out /kaggle/working/checkpoints/slot4_full/preds.csv

# 4. Полная оценка (metrics.json + таблица e1.csv по сплитам val, test_real, gold)
!python scripts/evaluate.py --dataset "$DATA_PATH" \
  --pred M0=/kaggle/working/m0.csv --pred M1=/kaggle/working/m1.csv --pred M2=/kaggle/working/checkpoints/slot4_full/preds.csv \
  --out_metrics /kaggle/working/checkpoints/slot4_full/metrics.json --out_table /kaggle/working/checkpoints/slot4_full/table_e1.csv \
  --figdir /kaggle/working/checkpoints/slot4_full/figures_e1
!head -n 25 /kaggle/working/checkpoints/slot4_full/table_e1.csv

## Сессия 2: Факториальное сравнение бэкбонов и контекста (Слоты 5–7)

In [ ]:
# ==============================================================================
# Слот 5: ModernBERT-base @1024 (контроль домена)
# ==============================================================================
# 1. Обучение модели
!torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml \
  --data_path "$DATA_PATH" --model_name answerdotai/ModernBERT-base \
  --max_len 1024 --train_mix full \
  --output_dir /kaggle/working/checkpoints/slot5_modernbert1024 --runs_csv reports/runs.csv

# 2. Очистка промежуточных чекпоинтов (сохранение дисковой квоты 20 ГБ)
!rm -rf /kaggle/working/checkpoints/slot5_modernbert1024/checkpoint-*
!ls -lh /kaggle/working/checkpoints/slot5_modernbert1024/best

# 3. Инференс на test_real и gold (проверка доменного сдвига E2 / вклада diff E3)
!python scripts/predict_m2.py --checkpoint /kaggle/working/checkpoints/slot5_modernbert1024/best \
  --dataset "$DATA_PATH"  --out /kaggle/working/checkpoints/slot5_modernbert1024/preds.csv

# 4. Полная оценка (metrics.json + таблица e1.csv по сплитам val, test_real, gold)
!python scripts/evaluate.py --dataset "$DATA_PATH" \
  --pred M0=/kaggle/working/m0.csv --pred M1=/kaggle/working/m1.csv --pred M2=/kaggle/working/checkpoints/slot5_modernbert1024/preds.csv \
  --out_metrics /kaggle/working/checkpoints/slot5_modernbert1024/metrics.json --out_table /kaggle/working/checkpoints/slot5_modernbert1024/table_e1.csv \
  --figdir /kaggle/working/checkpoints/slot5_modernbert1024/figures_e1
!head -n 25 /kaggle/working/checkpoints/slot5_modernbert1024/table_e1.csv

In [ ]:
# ==============================================================================
# Слот 6: GTE-ModernBERT-base @2048 (снятие усечения + relevance претрейн)
# ==============================================================================
# 1. Обучение модели
!torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml \
  --data_path "$DATA_PATH" --model_name Alibaba-NLP/gte-modernbert-base \
  --max_len 2048 --gradient_checkpointing --train_mix full \
  --output_dir /kaggle/working/checkpoints/slot6_gte2048 --runs_csv reports/runs.csv

# 2. Очистка промежуточных чекпоинтов (сохранение дисковой квоты 20 ГБ)
!rm -rf /kaggle/working/checkpoints/slot6_gte2048/checkpoint-*
!ls -lh /kaggle/working/checkpoints/slot6_gte2048/best

# 3. Инференс на test_real и gold (проверка доменного сдвига E2 / вклада diff E3)
!python scripts/predict_m2.py --checkpoint /kaggle/working/checkpoints/slot6_gte2048/best \
  --dataset "$DATA_PATH" --batch_size 16 --out /kaggle/working/checkpoints/slot6_gte2048/preds.csv

# 4. Полная оценка (metrics.json + таблица e1.csv по сплитам val, test_real, gold)
!python scripts/evaluate.py --dataset "$DATA_PATH" \
  --pred M0=/kaggle/working/m0.csv --pred M1=/kaggle/working/m1.csv --pred M2=/kaggle/working/checkpoints/slot6_gte2048/preds.csv \
  --out_metrics /kaggle/working/checkpoints/slot6_gte2048/metrics.json --out_table /kaggle/working/checkpoints/slot6_gte2048/table_e1.csv \
  --figdir /kaggle/working/checkpoints/slot6_gte2048/figures_e1
!head -n 25 /kaggle/working/checkpoints/slot6_gte2048/table_e1.csv

In [ ]:
# ==============================================================================
# Слот 7: ModernBERT-base @2048 (контроль контекста)
# Опциональный слот: если PR-AUC слота 6 не превосходит слот 5 на >= 0.005, ячейку можно пропустить для экономии ~50 мин GPU
# ==============================================================================
# 1. Обучение модели
!torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml \
  --data_path "$DATA_PATH" --model_name answerdotai/ModernBERT-base \
  --max_len 2048 --gradient_checkpointing --train_mix full \
  --output_dir /kaggle/working/checkpoints/slot7_modernbert2048 --runs_csv reports/runs.csv

# 2. Очистка промежуточных чекпоинтов (сохранение дисковой квоты 20 ГБ)
!rm -rf /kaggle/working/checkpoints/slot7_modernbert2048/checkpoint-*
!ls -lh /kaggle/working/checkpoints/slot7_modernbert2048/best

# 3. Инференс на test_real и gold (проверка доменного сдвига E2 / вклада diff E3)
!python scripts/predict_m2.py --checkpoint /kaggle/working/checkpoints/slot7_modernbert2048/best \
  --dataset "$DATA_PATH" --batch_size 16 --out /kaggle/working/checkpoints/slot7_modernbert2048/preds.csv

# 4. Полная оценка (metrics.json + таблица e1.csv по сплитам val, test_real, gold)
!python scripts/evaluate.py --dataset "$DATA_PATH" \
  --pred M0=/kaggle/working/m0.csv --pred M1=/kaggle/working/m1.csv --pred M2=/kaggle/working/checkpoints/slot7_modernbert2048/preds.csv \
  --out_metrics /kaggle/working/checkpoints/slot7_modernbert2048/metrics.json --out_table /kaggle/working/checkpoints/slot7_modernbert2048/table_e1.csv \
  --figdir /kaggle/working/checkpoints/slot7_modernbert2048/figures_e1
!head -n 25 /kaggle/working/checkpoints/slot7_modernbert2048/table_e1.csv

## Сессия 3: Финализация — Стэкинг, Абляция и train_best.yaml (Слоты 8–9)

In [ ]:
# ==============================================================================
# Слот 8: E2b — single-split стэкинг M1 + M2 на отложенных предсказаниях val_real
# ==============================================================================
import os, glob, subprocess, pandas as pd
df_runs = pd.read_csv('reports/runs.csv')
is_neural = ~df_runs['model_name'].astype(str).str.startswith('stacking')
is_not_ablation = ~df_runs.get('drop_diff', pd.Series(False, index=df_runs.index)).astype(str).str.lower().isin(['true', '1'])
neural_runs = df_runs.loc[is_neural & is_not_ablation].dropna(subset=['eval_pr_auc'])

def _has_ckpt(d):
    name = os.path.basename(str(d))
    return os.path.exists(os.path.join(str(d), 'best')) or os.path.exists(os.path.join(str(d), 'preds.csv')) or \
           bool(glob.glob(f'/kaggle/input/**/{name}/best', recursive=True)) or \
           bool(glob.glob(f'/kaggle/input/**/{name}/preds.csv', recursive=True))

neural_runs = neural_runs.loc[neural_runs['output_dir'].apply(_has_ckpt)]
best_run = neural_runs.sort_values(by=['eval_pr_auc', 'timestamp'], ascending=[False, False]).iloc[0]

best_ckpt = os.path.join(best_run['output_dir'], 'best')
best_preds = os.path.join(best_run['output_dir'], 'preds.csv')
ckpt_name = os.path.basename(best_run['output_dir'])
if not os.path.exists(best_ckpt):
    _cands = glob.glob(f'/kaggle/input/**/{ckpt_name}/best', recursive=True)
    if _cands: best_ckpt = _cands[0]
if not os.path.exists(best_preds):
    _cands_p = glob.glob(f'/kaggle/input/**/{ckpt_name}/preds.csv', recursive=True)
    if _cands_p: best_preds = _cands_p[0]

print(f"Лучший бэкбон для стэкинга: {best_run['model_name']} (PR-AUC: {best_run['eval_pr_auc']:.5f})")
print(f"Используемый источник M2: preds={best_preds} (exists={os.path.exists(best_preds)}), ckpt={best_ckpt}")

data_path = os.environ.get('DATA_PATH', 'data/processed/dataset_v1.parquet')
if os.path.exists(best_preds):
    cmd_stack = (
        f'python scripts/train_stacking.py --dataset "{data_path}" '
        f'--m1_preds /kaggle/working/m1.csv --m2_preds "{best_preds}" '
        f'--output_dir /kaggle/working/checkpoints/slot8_stacking '
        f'--out_preds /kaggle/working/checkpoints/slot8_stacking/m_stack.csv '
        f'--runs_csv reports/runs.csv'
    )
else:
    cmd_stack = (
        f'python scripts/train_stacking.py --dataset "{data_path}" '
        f'--m1_preds /kaggle/working/m1.csv --m2_checkpoint "{best_ckpt}" --batch_size 16 '
        f'--output_dir /kaggle/working/checkpoints/slot8_stacking '
        f'--out_preds /kaggle/working/checkpoints/slot8_stacking/m_stack.csv '
        f'--runs_csv reports/runs.csv'
    )
subprocess.run(cmd_stack, shell=True, check=True)

# Оценка стэкинга через evaluate.py
!python scripts/evaluate.py --dataset "$DATA_PATH" \
  --pred M0=/kaggle/working/m0.csv --pred M1=/kaggle/working/m1.csv \
  --pred M2=/kaggle/working/checkpoints/slot8_stacking/m_stack.csv \
  --out_metrics /kaggle/working/checkpoints/slot8_stacking/metrics.json \
  --out_table /kaggle/working/checkpoints/slot8_stacking/table_e1.csv \
  --figdir /kaggle/working/checkpoints/slot8_stacking/figures_e1
!head -n 25 /kaggle/working/checkpoints/slot8_stacking/table_e1.csv


In [ ]:
# ==============================================================================
# Слот 9: E3 — Абляция БЕЗ diff_code (--drop_diff) на архитектуре лучшего бэкбона
# ==============================================================================
if 'best_run' not in locals():
    import os, glob, pandas as pd
    df_runs = pd.read_csv('reports/runs.csv')
    is_neural = ~df_runs['model_name'].astype(str).str.startswith('stacking')
    is_not_ablation = ~df_runs.get('drop_diff', pd.Series(False, index=df_runs.index)).astype(str).str.lower().isin(['true', '1'])
    neural_runs = df_runs.loc[is_neural & is_not_ablation].dropna(subset=['eval_pr_auc'])
    def _has_ckpt(d):
        name = os.path.basename(str(d))
        return os.path.exists(os.path.join(str(d), 'best')) or os.path.exists(os.path.join(str(d), 'preds.csv')) or \
               bool(glob.glob(f'/kaggle/input/**/{name}/best', recursive=True)) or \
               bool(glob.glob(f'/kaggle/input/**/{name}/preds.csv', recursive=True))
    neural_runs = neural_runs.loc[neural_runs['output_dir'].apply(_has_ckpt)]
    best_run = neural_runs.sort_values(by=['eval_pr_auc', 'timestamp'], ascending=[False, False]).iloc[0]

best_model_name = best_run["model_name"]
best_max_len = int(best_run["max_len"])
gc_flag = "--gradient_checkpointing" if best_max_len >= 2048 else ""

data_path = os.environ.get('DATA_PATH', 'data/processed/dataset_v1.parquet')
cmd_train = (
    f'torchrun --nproc_per_node=2 scripts/train_m2.py --config config/train.yaml '
    f'--data_path "{data_path}" --model_name "{best_model_name}" '
    f'--max_len {best_max_len} {gc_flag} --drop_diff --train_mix full '
    f'--output_dir /kaggle/working/checkpoints/slot9_ablation_no_diff --runs_csv reports/runs.csv'
)
import subprocess
print(f"Выполнение: {cmd_train}")
subprocess.run(cmd_train, shell=True, check=True)
!rm -rf /kaggle/working/checkpoints/slot9_ablation_no_diff/checkpoint-*

# Инференс с флагом --drop_diff
!python scripts/predict_m2.py --checkpoint /kaggle/working/checkpoints/slot9_ablation_no_diff/best \
  --dataset "$DATA_PATH" --batch_size 16 --drop_diff --out /kaggle/working/checkpoints/slot9_ablation_no_diff/preds.csv

# Оценка абляции
!python scripts/evaluate.py --dataset "$DATA_PATH" \
  --pred M0=/kaggle/working/m0.csv --pred M1=/kaggle/working/m1.csv \
  --pred M2=/kaggle/working/checkpoints/slot9_ablation_no_diff/preds.csv \
  --out_metrics /kaggle/working/checkpoints/slot9_ablation_no_diff/metrics.json \
  --out_table /kaggle/working/checkpoints/slot9_ablation_no_diff/table_e1.csv \
  --figdir /kaggle/working/checkpoints/slot9_ablation_no_diff/figures_e1
!head -n 25 /kaggle/working/checkpoints/slot9_ablation_no_diff/table_e1.csv


In [ ]:
# ==============================================================================
# Финализация: автоматический выбор лучшего бэкбона по val_real PR-AUC
# ==============================================================================
!python scripts/finalize_best_config.py --runs_csv reports/runs.csv \
  --out_config config/train_best.yaml
!cat config/train_best.yaml

import pandas as pd
df_runs = pd.read_csv("reports/runs.csv")
print("\n=== ИТОГОВАЯ ТАБЛИЦА ВСЕХ ПРОГОНОВ FD-24 ===")
print(df_runs.to_string(index=False))